# Avaliação do Modelo THERA (BioGPT Fine-Tuned)

## Objetivo

O objetivo central deste notebook é avaliar o modelo **THERA**, criado a partir do **fine-tuning do BioGPT**. A ideia principal é:

- Separar o **conjunto de testes** da base de dados.
- Utilizar um **pipeline de avaliação híbrido**, combinando métricas automáticas (BERTScore, F1 e acurácia, completude) e avaliação baseada em "LLM-as-a-judge".

---

## Metodologia

1. **Separação do conjunto de testes**
   - Garantir que os casos de teste não foram usados durante o fine-tuning.
   - Cada entrada consiste em uma lista de sintomas, com saída esperada de:
     - Diagnóstico
     - Descrição médica da doença
     - Fatores de risco

2. **Execução do pipeline**
   - Rodar o modelo **THERA** sobre o conjunto de testes.
   - Comparar com:
     - **Modelo base** (BioGPT) com prompt ajustado, se possível.
     - **Outro modelo instruction-tuned** (MedAlpaca, GPT-4 local) para comparação de qualidade clínica.
   
3. **Limpeza e normalização de saída**
   - Separar **Diagnosis, Description, Risk Factors**.
   - Limpar duplicatas e caracteres estranhos.
   - Estruturar fatores de risco em **bullet points**.

4. **Métricas de avaliação**
   - **Completeness Rate**: % de campos preenchidos corretamente.
   - **BERTScore**: Similaridade entre Description e Risk Factors com referência.
   - **Exact Match / F1**: Comparação de Diagnosis com gabarito.
   - **Avaliação subjetiva**: LLM-as-a-judge com critérios de aceitação e uma pontuação numérica.

5. **Análise e visualização**
   - Gerar tabelas comparativas entre modelos.
   - Identificar pontos fortes e fracos do fine-tuning.
   - Destacar melhorias em relação ao modelo base e relevância clínica.

---

## Considerações Finais

- A avaliação híbrida permite **quantificar e qualificar** o desempenho do modelo THERA.
- A limpeza textual e padronização de outputs é essencial para garantir **coerência e legibilidade**.
- Comparações com outros modelos instruídos destacam o **valor real do fine-tuning**, mostrando ganhos em completude, coerência e qualidade clínica.


## Configuração dos Modelos e Preparação dos Dados

A metodologia de preparação dos dados segue a lógica do treinamento, consistindo em preparar as instâncias em pares de input (sintomas) e output (diagnóstico, descrição e fatores de risco). 

In [1]:
import pandas as pd
import torch
from transformers import BioGptTokenizer, BioGptForCausalLM, pipeline
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from openai import OpenAI
import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import re

c:\Users\mario\.conda\envs\medical_llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------
# 🔧 Configurações
# ---------------------------
MODEL_NAME = "microsoft/biogpt"
TRAINED_MODEL_DIR = "thera-finetuned-symptom-diagnosis"
MAX_LENGTH = 512
BATCH_SIZE = 4
DEVICE = 0 if torch.cuda.is_available() else -1

# LLM-as-a-Judge
openai_client = OpenAI(api_key="YOUR_OPENAI_API_KEY")  # Coloque sua chave aqui

In [4]:
# ---------------------------
# 📥 Carregamento e preparação do dataset
# ---------------------------
df = pd.read_csv("./data/merged_dataset.csv")
df.drop(columns=["Unnamed: 0"], inplace=True)
COLUNAS = df.columns

def gerar_pares(row):
    sintomas = [col.replace("_", " ") for col in COLUNAS if row[col] == 1]
    input_text = f"Given the following symptoms, provide the most likely diagnosis, description, and risk factors.\n\nSymptoms: {', '.join(sintomas)}."
    output_text = f'''
        Diagnosis: {row['diseases']}.
        Description: {row['diseases_description']}.
        Risk factors: {row['disease_risk_factors']}.
    '''
    return {"input": input_text.strip(), "output": output_text.strip()}

caso_diagnostico = df.apply(gerar_pares, axis=1).tolist()
dataset = Dataset.from_list(caso_diagnostico)
eval_dataset = dataset.train_test_split(test_size=0.15)["test"]

In [3]:
# ---------------------------
# 🔠 Carregar modelos e tokenizers
# ---------------------------
tokenizer_base = BioGptTokenizer.from_pretrained(MODEL_NAME)
model_base = BioGptForCausalLM.from_pretrained(MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_trained = BioGptTokenizer.from_pretrained(TRAINED_MODEL_DIR)
model_trained = BioGptForCausalLM.from_pretrained(TRAINED_MODEL_DIR).to("cuda" if torch.cuda.is_available() else "cpu")

generator_base = pipeline('text-generation', model=model_base, tokenizer=tokenizer_base, device=DEVICE)
generator_trained = pipeline('text-generation', model=model_trained, tokenizer=tokenizer_trained, device=DEVICE)

## Geração dos Casos no Conjunto de Teste

O casos separados no conjunto de teste serão utilizados para geração de diagnósticos pelo modelo treinado e o modelo já instruído. Os resultados irão passar pelo pipleine híbrido de avaliação para extração de métricas e avaliação por LLM-as-a-judge.

In [4]:
def clean_medical_output(raw_text):
    # Extrair Diagnosis, Description, RiskFactors
    def truncate_at_last_period(text):
        if "." in text:
            return text[:text.rfind(".")+1]
        return text
    raw_text = truncate_at_last_period(raw_text)
    diagnosis = re.search(r"Diagnosis:\s*(.*?)(?:Description:|Risk factors:|$)", raw_text, re.I | re.S)
    description = re.search(r"Description:\s*(.*?)(?:Risk factors:|$)", raw_text, re.I | re.S)
    risk = re.search(r"Risk factors:\s*(.*)", raw_text, re.I | re.S)

    diagnosis = diagnosis.group(1).strip() if diagnosis else ""
    description = description.group(1).strip() if description else ""
    risk_text = risk.group(1).strip() if risk else ""

    # Limpar Risk Factors
    risk_text = re.sub(r'\.{2,}|\(\.\.\)|\)', '.', risk_text)  # remover parênteses estranhos
    # Quebrar em frases
    phrases = re.split(r'\.|\n', risk_text)
    phrases = [p.strip().capitalize() for p in phrases if p.strip()]
    # Remover duplicatas mantendo ordem
    seen = set()
    clean_phrases = []
    for p in phrases:
        if p not in seen:
            clean_phrases.append(p)
            seen.add(p)
    # Transformar em bullet points
    clean_risk = "\n- ".join(clean_phrases)
    if clean_risk:
        clean_risk = "- " + clean_risk

    return {
        "Diagnosis": diagnosis,
        "Description": description,
        "RiskFactors": clean_risk
    }

**Exemplo de geração do modelo treinado**

In [5]:
inp = "Given the following symptoms, provide the most likely diagnosis, description, and risk factors.\n\nSymptoms: decrease in appetite, nasal congestion (finding), pulling at ears, shortness of breath."
resp_trained = generator_trained(inp, max_length=300, truncation=True)[0]["generated_text"]
print("\nTrained Model Response:\n", clean_medical_output(resp_trained))

c:\Users\mario\.conda\envs\medical_llm\lib\site-packages\transformers\models\biogpt\modeling_biogpt.py:330: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(



Trained Model Response:
 {'Diagnosis': 'Acute bronchiolitis', 'Description': 'Acute inflammation of the bronchioles usually caused by the respiratory syncytial virus.', 'RiskFactors': '- Young age, poor feeding of babies, poor weight gain, and respiratory tract infections in early life, such as rhinovirus infection\n- Having a long-term illness or weakened immune system\n- Having a weakened immune system or another condition such as hiv / aids, diabetes or asthma\n- Having a history of alcohol, drug or tobacco use\n- Having a history of being around smoke or strong odors\n- Having a mother who smoked during the first year of life\n- Being older than 40 years\n- Being born prematurely\n- Low birth weight babies are more likely to get bronchiolitis in early life, particularly if they were born prematurely\n- Low birth weight also makes it more likely to get bronchiolitis\n- Having a mother with diabetes mellitus or a weakened immune system'}


**Inferência em lote no conjunto de teste**

In [ ]:
# ---------------------------
# 🧪 Inferência em lote
# ---------------------------
inputs = [x["input"] for x in eval_dataset]
y_true = [x["output"] for x in eval_dataset]

results_base, results_trained = [], []

for inp in tqdm.tqdm(inputs, desc="Gerando respostas"):
    res_base = generator_base(inp, max_length=150)[0]["generated_text"]
    res_trained = generator_trained(inp, max_length=150)[0]["generated_text"]
    results_base.append(res_base)
    results_trained.append(res_trained)

**Extração dos valores de diagnóstico dos modelos**

In [ ]:
# ---------------------------
# 📌 Função de extração de diagnóstico
# ---------------------------
def extrair_diagnostico(text):
    if "Diagnosis:" in text:
        return text.split("Diagnosis:")[1].split(".")[0].strip()
    return text

y_true_diag = [extrair_diagnostico(x) for x in y_true]
y_base_diag = [extrair_diagnostico(x) for x in results_base]
y_trained_diag = [extrair_diagnostico(x) for x in results_trained]

## Avaliação dos Resultados

In [ ]:
# ---------------------------
# 📊 Métricas clássicas
# ---------------------------
acc_base = accuracy_score(y_true_diag, y_base_diag)
acc_trained = accuracy_score(y_true_diag, y_trained_diag)
f1_base = f1_score(y_true_diag, y_base_diag, average='macro')
f1_trained = f1_score(y_true_diag, y_trained_diag, average='macro')

print("📊 Resultados métricas clássicas:")
print(f"Base - Acurácia: {acc_base:.3f}, F1-score: {f1_base:.3f}")
print(f"Treinado - Acurácia: {acc_trained:.3f}, F1-score: {f1_trained:.3f}")

In [ ]:
# ---------------------------
# ⚡ Perplexidade
# ---------------------------
def calcular_perplexidade(model, tokenizer, texts):
    model.eval()
    perplexities = []
    for text in tqdm.tqdm(texts, desc="Calculando perplexidade"):
        encodings = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
        input_ids = encodings.input_ids.to(model.device)
        with torch.no_grad():
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
        perplexity = torch.exp(loss).item()
        perplexities.append(perplexity)
    return perplexities

perp_base = calcular_perplexidade(model_base, tokenizer_base, [x["input"] + "\n" + x["output"] for x in eval_dataset])
perp_trained = calcular_perplexidade(model_trained, tokenizer_trained, [x["input"] + "\n" + x["output"] for x in eval_dataset])

In [ ]:
# ---------------------------
# 💡 LLM-as-a-Judge
# ---------------------------
def judge_responses(case_text, base_resp, trained_resp):
    prompt = f"""
        Você é um especialista clínico. Avalie as respostas abaixo para o caso:
        {case_text}

        Resposta modelo base: {base_resp}
        Resposta modelo treinado: {trained_resp}

        Critérios:
        1. Coerência com os sintomas
        2. Relevância do diagnóstico
        3. Argumentação baseada em literatura médica

        Dê uma pontuação de 0 a 10 para cada resposta, indique qual é melhor e se há alguma alucinação (afirmação médica incorreta).
        Formato de resposta:
        Modelo Base: X
        Modelo Treinado: Y
        Melhor Resposta: Base ou Treinado
        Alucinação Base: Sim/Não
        Alucinação Treinado: Sim/Não
    """
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

judge_results = []
for case, base_resp, trained_resp in tqdm.tqdm(zip(inputs, results_base, results_trained), total=len(inputs), desc="LLM-as-a-Judge"):
    try:
        result = judge_responses(case, base_resp, trained_resp)
    except Exception as e:
        result = f"Erro: {e}"
    judge_results.append(result)

In [ ]:
# ---------------------------
# 💾 Salvar resultados
# ---------------------------
df_results = pd.DataFrame({
    "case": inputs,
    "true_output": y_true,
    "base_model_output": results_base,
    "trained_model_output": results_trained,
    "base_diag": y_base_diag,
    "trained_diag": y_trained_diag,
    "perplexity_base": perp_base,
    "perplexity_trained": perp_trained,
    "judge_evaluation": judge_results
})
df_results.to_csv("./data/evaluation_results_complete.csv", index=False)
print("✅ Avaliação completa salva em 'evaluation_results_complete.csv'")

In [ ]:
# ---------------------------
# 📊 Gráficos automáticos
# ---------------------------

# 1️⃣ Métricas clássicas
metrics = ["Acurácia", "F1-score"]
base_scores = [acc_base, f1_base]
trained_scores = [acc_trained, f1_trained]

x = range(len(metrics))
plt.figure(figsize=(8,6))
plt.bar([i-0.15 for i in x], base_scores, width=0.3, label="Modelo Base")
plt.bar([i+0.15 for i in x], trained_scores, width=0.3, label="Modelo Treinado")
plt.xticks(x, metrics)
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("Comparação de métricas clássicas")
plt.legend()
plt.grid(axis='y')
plt.show()

# 2️⃣ Distribuição de diagnósticos corretos
df_results["base_correct"] = df_results["base_diag"] == df_results["true_output"].apply(extrair_diagnostico)
df_results["trained_correct"] = df_results["trained_diag"] == df_results["true_output"].apply(extrair_diagnostico)

plt.figure(figsize=(8,6))
sns.countplot(data=df_results.melt(value_vars=["base_correct","trained_correct"],
                                   var_name="Modelo", value_name="Correto"),
              x="Modelo", hue="Correto")
plt.title("Distribuição de diagnósticos corretos")
plt.ylabel("Número de casos")
plt.show()

# 3️⃣ Decisão do LLM-as-a-Judge
def extract_best(judge_text):
    if "Melhor Resposta: Base" in judge_text:
        return "Base"
    elif "Melhor Resposta: Treinado" in judge_text:
        return "Treinado"
    else:
        return "Indefinido"

df_results["judge_best"] = df_results["judge_evaluation"].apply(extract_best)

plt.figure(figsize=(6,6))
sns.countplot(data=df_results, x="judge_best", palette="Set2")
plt.title("Decisão do LLM-as-a-Judge")
plt.ylabel("Número de casos")
plt.show()

# 4️⃣ Distribuição de pontuações
def extract_scores(judge_text):
    base_score = re.search(r"Modelo Base:\s*(\d+)", judge_text)
    trained_score = re.search(r"Modelo Treinado:\s*(\d+)", judge_text)
    return int(base_score.group(1)) if base_score else None, int(trained_score.group(1)) if trained_score else None

df_results[["score_base","score_trained"]] = df_results["judge_evaluation"].apply(lambda x: pd.Series(extract_scores(x)))

plt.figure(figsize=(10,5))
sns.histplot(df_results, x="score_base", color="blue", label="Base", kde=True, binwidth=1)
sns.histplot(df_results, x="score_trained", color="orange", label="Treinado", kde=True, binwidth=1)
plt.title("Distribuição de pontuações do LLM-as-a-Judge")
plt.xlabel("Pontuação")
plt.ylabel("Número de casos")
plt.legend()
plt.show()

# 5️⃣ Distribuição de perplexidade
plt.figure(figsize=(10,5))
sns.histplot(df_results, x="perplexity_base", color="blue", label="Base", kde=True)
sns.histplot(df_results, x="perplexity_trained", color="orange", label="Treinado", kde=True)
plt.title("Distribuição de perplexidade por modelo")
plt.xlabel("Perplexidade")
plt.ylabel("Número de casos")
plt.legend()
plt.show()